In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown ,display


In [ ]:
load_dotenv(override=True)

In [ ]:
open_router_key=os.getenv('OPENROUTER_API_KEY')
gemini_key=os.getenv('GEMINI_API_KEY')

if open_router_key:
    print(f'Open API Key exisits and begins with {open_router_key[0:5]}')
else:
    print("OpenAPI Key not set")

if gemini_key:
    print(f"Gemini key exist it starts with {gemini_key[0:5]}")
else:
    print("GEmini  key does not exist")

In [ ]:
Job_descpription="""

Junior Software Developer Responsibilities:
->Assisting the development manager with all aspects of software design and coding.
->Attending and contributing to company development meetings.
->Learning the codebase and improving your coding skills.
->Writing and maintaining code.
->Working on minor bug fixes.
->Monitoring the technical performance of internal systems.
->Responding to requests from the development team.
->Gathering information from consumers about program functionality.
->Writing reports.
->Conducting development tests.

"""


In [ ]:
bullet_points="""
->good at problem solving
->have knowlded for python devlopement and backend devlopment
->worked as qa anyalst for fintech
->tested the big code base and able to find the error in between
->good at writing reports
->eagar to learn and imporve coding skill
->good and montotring and mainting the devlopment process

"""

In [ ]:

status='FAIL'
max_iter=5
iteration=0
feed_back="No feedback yet waiting for the first draft"

In [ ]:

while status!='PASS' and max_iter>iteration:
    iteration+=1
    optimizer_prompt=f"""
    Job_description:{Job_descpription}
    current_draft_resume:{bullet_points}
    feed_back={feed_back}

    your job is to write the resume which matchs the job description 
    and have the feed back implementation so that the user can get the best job job and have higher chance of getting the job based on this bullet points
    NOTE: have the ouput in the bullet points format only

    
    """
    optimizer=OpenAI(api_key=gemini_key,base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    model_name="gemini-2.5-flash"
    message=[{'role':'user','content':optimizer_prompt}]
    response=optimizer.chat.completions.create(model=model_name,messages=message)
    current_draft=response.choices[0].message.content.strip()
    bullet_points=current_draft

    evaluator_prompt=evaluator_prompt = f"""
    You are a strict technical recruiter. Default to FAIL unless the resume is interview-ready.
    
    Job description:
    {Job_descpription}
    
    Resume bullets:
    {current_draft}
    
    Score each criterion 1-10. PASS only if ALL are true:
    1. Every major job responsibility appears in at least one bullet (with evidence, not vague claims)
    2. Bullets use strong action verbs and quantifiable results where possible
    3. No grammar/spelling errors
    4. Bullets are concise (one line each), not paragraph-style
    5. A hiring manager would shortlist this candidate for a Junior Software Developer role
    
    Be harsh. "Looks related" is FAIL. Only PASS if you would genuinely invite this candidate to interview.
    
    Respond EXACTLY in this format (no extra text):
    status: PASS or FAIL
    feed_back: specific improvements needed (required even if PASS)
    score: X/10
    """
   
    
    evaluator=OpenAI(base_url='http://localhost:11434/v1',api_key='ollama')
    model_name="llama3.2"
    message2=[{'role':'user','content':evaluator_prompt}]
    response1=evaluator.chat.completions.create(model=model_name,messages=message2)
    evaluation=response1.choices[0].message.content.strip()
    try:
        lines=evaluation.split("\n")
        status = lines[0].replace("status:", "").strip()
        feed_back = lines[1].replace("feed_back:", "").strip()
    except Exception:
        # Fallback if the LLM format breaks slightly
        status = 'FAIL'
        feed_back = "Please have it match the Job description."
    print(f"Evaluator Status: {status}")
    print(f"Evaluator Feedback: {feed_back}\n")

# Final execution results
print("--- 🏁 LOOP FINISHED ---")
if status =='PASS':
    print(f"✅ Success! Approved Copy: {bullet_points}")
else:
    print(f"⚠️ Hit maximum iterations. Best Copy: {bullet_points}")




